# Quantum Neural Network Example on NASA Bearing Dataset
This notebook demonstrates an example of running a quantum model on the NASA Bearing Dataset using Golomb encoding. Its main objective is to showcase how Quantum Neural Networks (QNNs) can be applied to real-world data for tasks such as fault detection and remaining useful life (RUL) prediction.



In [6]:
# Configuration
WINDOW_SIZE = 1
STRIDE = 1
FEATURE_NAMES = ['RMS_Bearing_1','RMS_Bearing_2','RMS_Bearing_3','RMS_Bearing_4']

ENCODING = 'hamming'
R = 2
L = 2

METRICS_OUTPUT_FILE = "test_metrics_output.txt"
MODEL_OUTPUT_FILE = 'test_qnn_output.json'

### 1. Data Exploration
- **Loading the Data:** The notebook starts by loading the NASA Bearing Dataset.

In [4]:
import pandas as pd
from NASA.Utils import create_X_Y_set, create_train_and_test_set, append_to_json
df_rms = pd.read_csv('df_rms.csv')

X,y = create_X_Y_set(df_rms, window_size=WINDOW_SIZE,stride=STRIDE,feature_names=FEATURE_NAMES)
print(f"Data shapes - X: {X.shape}, y: {y.shape}")

Data shapes - X: (983, 1), y: (983,)


## Train/Test Split Normalization Strategy

Consider implementing a two-step normalization process to enhance data preprocessing:

1. **Z-Score Normalization:**
   - Centers the data around a mean of 0 with unit variance.
   - Helps standardize features with different scales and distributions.

2. **Min-Max Scaling:**
   - Transforms the data to a fixed range, typically [0,1] or [-1,1].
   - Aligns well with the expected output range, improving model performance.

This approach ensures consistent feature scaling and improves model stability during training.


In [5]:
X_train, X_test, y_train, y_test = create_train_and_test_set(X, y, test_size=0.2)


Balancing Train data: [0.61450382 0.38549618]
Balancing Test data: [0.6142132 0.3857868]


### 2. Quantum Model Execution

- **Model Setup:** Example code is provided to run a quantum model (QNN) on the dataset.

In [7]:
from qnn import QNN
from qnn.constants import BINARY_CROSS_ENTROPY
qnn = QNN(
    R=R,
    L=L,
    N=X_train.shape[1],
    ansatz='sequential',
    encoding=ENCODING,
    loss_fn=BINARY_CROSS_ENTROPY,
    trainable_block_layers=5,
    save_weights=True,
    save_losses=True,
    seed=42,
    max_iter=100,
    step_size=0.001,
    verbose=True
)

qnn.fit(X_train, y_train)

weights shape: (3, 5, 2, 3), size: 0.00 MB
Checking loss_fn behavior...
Loss function result: 1.4987510725918982


Best loss: 0.355383: 100%|██████████| 100/100 [00:05<00:00, 17.66it/s]


## Model Evaluation: Training and Test Performance

To assess the performance of the Quantum Neural Network (QNN), we evaluate key classification metrics on both the **training** and **test** datasets. This ensures the model generalizes well and is not overfitting to the training data.

### 1. Training Set Evaluation
- Predictions are made on the training data, thresholding at 0.5.
- **Accuracy** is calculated to measure overall classification correctness.
- A **confusion matrix** is generated to visualize classification errors.

### 2. Test Set Evaluation
For the test dataset, we compute multiple performance metrics:
- **Accuracy**: Percentage of correctly classified samples.
- **Precision**: Measures how many predicted positives are actual positives.
- **Recall**: Measures how many actual positives were correctly predicted.
- **F1-Score**: Harmonic mean of precision and recall, balancing both metrics.
- **ROC-AUC Score**: Evaluates the ability to distinguish between classes, computed using prediction probabilities.
- **Confusion Matrix**: Provides a detailed breakdown of true positives, false positives, true negatives, and false negatives.

### Summary of Output
- The training accuracy is printed as a percentage.
- A confusion matrix is displayed for both the training and test sets.
- The test set metrics are computed and printed in a structured format.

This evaluation process provides a comprehensive understanding of the model's classification performance, helping to determine if further tuning or adjustments are required.


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# --------------------------------
# Evaluate on the training set
# ---------------------------------
y_train_pred = qnn.predict(X_train) > 0.5
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
train_conf_matrix = confusion_matrix(y_train, y_train_pred)
print(f"Train Confusion matrix {train_conf_matrix}")

# ---------------------------------
# Evaluate on the test set
# ---------------------------------
y_test_pred = qnn.predict(X_test) > 0.5
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, qnn.predict(X_test))  # Use probabilities for ROC-AUC
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

print(f"Test Metrics:\n")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"    Precision: {test_precision:.4f}\n")
print(f"    Recall: {test_recall:.4f}\n")
print(f"    F1-Score: {test_f1:.4f}\n")
print(f"    ROC-AUC: {test_roc_auc:.4f}\n")
print(f"    Test Confusion matrix {test_conf_matrix}")


Training Accuracy: 84.10%
Train Confusion matrix [[483   0]
 [125 178]]
Test Metrics:

Test Accuracy: 83.76%
    Precision: 1.0000

    Recall: 0.5789

    F1-Score: 0.7333

    ROC-AUC: 1.0000

    Test Confusion matrix [[121   0]
 [ 32  44]]


## Saving Model Configuration and Training Progress

To preserve the trained Quantum Neural Network (QNN) model's configuration and training history, we store key parameters and performance metrics in a structured JSON file. This allows for future model analysis, reproducibility, and comparison with alternative configurations.



In [15]:
model_output_file = MODEL_OUTPUT_FILE
new_data = {
    "Model_Configuration": {
        "R": qnn.R,
        "L": qnn.L,
        "Encoding": qnn.encoding,
        "Step_Size": qnn.step_size,
        "Max_Iter": qnn.max_iter,
        "Trained_weights_": qnn.trained_weights_.tolist(),
        "Loss_Values": [float(loss) for loss in qnn.losses],  # Loss values during training
    },

}
append_to_json(model_output_file, new_data)
print(f"Data appended to {model_output_file}")


Data appended to test_qnn_output.json
Data appended to test_qnn_output.json
